In [25]:
import pandas as pd
import numpy as np

from pathlib import Path

In [26]:
%run ../gs_slimming.py
print("="*120)
print("gs_slimming done.")
print("="*120)
%run flattening.py
print("="*120)
print("flattening done.")
print("="*120)

Mismatches: {'ViacomCBS_ESG Report_2020-2021_vFINAL.pdf'}

No. of gold_standard reports: 139

No. of downloadable reports: 113

No. of reports in gs_slim: 54


status
partial     45
complete     9
Name: count, dtype: int64

scopes_present
[1, 2lb]            18
[1, 2lb, 3]         15
[1, 2lb, 2mb, 3]     9
[1, 2lb, 2mb]        6
[1, 2mb, 3]          1
[1, 3]               1
[1]                  1
[2lb, 3]             1
b]                1
[3]                  1
Name: count, dtype: int64
gs_slimming done.

  Flattening 53 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv
  Reports    : 52
  Zeilen     : 643
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv

Total Errors: 0
[OK] No errors detected.
flattening done.


## Import slimmed down Gold-Standard

In [27]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [28]:
answers= pd.read_csv("./PipelineA-Answers.csv")


df_to_merge = [
    (answers, "_A"),
]

In [ ]:
import re

def build_report_name_mapping(gs_names: list[str]) -> dict[str, str]:
    """
    Returns {normalized_name -> canonical_gs_name}.

    The pipeline sanitizes filenames by replacing every character that is not
    alphanumeric, underscore, or hyphen with an underscore — applying the same
    rule to GS names produces the keys that appear in `answers`.
    """
    mapping = {}
    for name in gs_names:
        key = re.sub(r"[^a-zA-Z0-9_-]", "_", name)
        if key in mapping:
            raise ValueError(
                f"Collision: {repr(name)} and {repr(mapping[key])} both normalize to {repr(key)}"
            )
        mapping[key] = name
    return mapping


gs_name_mapping = build_report_name_mapping(gs["report_name"].unique().tolist())

answers["report_name"] = answers["report_name"].map(
    lambda n: gs_name_mapping.get(n, n)  # fallback: keep original if no GS match
)

unmapped = sorted(set(answers["report_name"]) - set(gs["report_name"]))
print(f"Unmapped after normalization: {len(unmapped)}", "=> OK" if not unmapped else "")
for r in unmapped:
    print(" ", r)

In [29]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""
    label = raw.strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_later_year"  # 2021/2022 -> 2021
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [30]:
# Get years from extractions 
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynrom DataFrames to preserve the original ones
answers_ynorm = answers.copy()

df_to_merge_ynorm = [
    (answers_ynorm, "_A")
]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

In [31]:
#print("Think ok:", all(answers["report_name"].unique() == gs["report_name"].unique()))
#print()

#print(answers["report_name"].isin(gs["report_name"]).all())
#print(gs["report_name"].isin(answers["report_name"]).all())

in_think1_not_gs  = sorted(set(answers["report_name"]) - set(gs["report_name"]))
in_gs_not_answers  = sorted(set(gs["report_name"])     - set(answers["report_name"]))

print(f"In answers, nicht in GS: {len(in_think1_not_gs)} ==> {"OK" if len(in_think1_not_gs) == 0 else "NOK"}")
for r in sorted(in_think1_not_gs): print(" ", r)

print(f"\nIn GS, nicht in answers: {len(in_gs_not_answers)} ==> {"OK" if len(in_gs_not_answers) == 0 else "NOK"}")
for r in sorted(in_gs_not_answers): print(" ", r)

In answers, nicht in GS: 39 ==> NOK
  Fresenius_SE_2019_report
  ViacomCBS_ESG_Report_2020-2021_vFINAL
  acuity_brands_inc_2022_report
  allfunds_group_2021_report
  americold_realty_inc_2022_report
  atlas_arteria_2019_report
  autoneum_holding_2019_report
  bekaert__d__sa_2022_report
  blackline_safety_2021_report
  cabot_corp_2018_report
  cardinal_energy_ltd_2021_report
  comerica_inc_2019_report
  cytokinetics_inc_2022_report
  dampskibsselskabet_norden_2019_report
  evolution_mining_ltd_2020_report
  georg_fischer_ag_2018_report
  granite_construction_inc_2020_report
  h__lundbeck_2021_report
  hammerson_reit_2021_report
  hang_lung_properties_2018_report
  healius_ltd_2022_report
  hudbay_minerals_inc_2020_report
  inchcape_plc_2022_report
  jetblue_airways_corp_2019_report
  jpmorgan_chase_2020_report
  kilroy_realty_2017_report
  kilroy_realty_2018_report
  kureha_corp_2020_report
  lundin_gold_inc_2021_report
  lundin_gold_inc_2022_report
  metro_ag_2022_report
  morinaga_ltd

## Matching Extractions to Gold-Standard Scheme

### Before ynrom

In [ ]:
years = set()
year_report = []

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

### After ynrom

In [ ]:
years = set()
year_report = []

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

## Mapping special occurences

In [ ]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



In [ ]:

# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



## Merging Extraction Values and Gold-Standard

In [ ]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [ ]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

In [ ]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [ ]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [ ]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

## Saving created DataFrame

In [ ]:
merged.to_json("PipelineA.json", index=False, orient="records", indent=4)
merged_ynorm.to_json("PipelineA_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable
